# Manual review: resolving flagged accusation targets

Walks through `flagged_for_review.csv` one row at a time, printing the surrounding
dialogue (10 lines either side) with the flagged line marked `>>>`. Other lines in
that window that were themselves flagged for annotation carry a trailing `*`, so
it's visible when several accusations sit close together.

For each row you can:

- type one or more player names, comma-separated (e.g. `Elliot, Sian`) → resolves the target and writes it directly into the annotation JSON
- press Enter (empty input) or type `u` → **confirms** the target is genuinely unresolvable; stays `UNKNOWN`, but is marked as manually reviewed so you know it's been looked at, not just skipped
- type `s` → skip this row for now (comes back next time you run the loop)
- type `q` → stop the loop early; everything done so far is already saved

Widen or narrow the window with `CONTEXT_BEFORE` / `CONTEXT_AFTER` in the config
cell. If the target still isn't resolvable from the printed window, open the full
transcript at `ready_for_annotation/<source>/<...>.txt` and jump to the line number.

Progress is saved after every single answer (both to the CSV and to the annotation JSON), so it's safe to stop and resume anytime.


In [ ]:
import json
import csv
import os
from pathlib import Path

# ---- adjust these if your paths differ ----
CSV_PATH = Path(
    r"C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\processed"
    r"\lai2023\accusation_transcripts\acc_targets\flagged_for_review.csv"
)
OUTPUT_ROOT = Path(
    r"C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\processed"
    r"\lai2023\accusation_transcripts\acc_targets"
)
TRANSCRIPT_ROOT = Path(
    r"C:\Users\annab\Documents\GitHub\masters_thesis_sdg\data\processed"
    r"\lai2023\accusation_transcripts\ready_for_annotation"
)
CONTEXT_BEFORE = 10
CONTEXT_AFTER = 10
# --------------------------------------------

FIELDNAMES_EXTRA = ["review_status", "resolved_accused"]

MARKER = "<<ACCUSATION_TO_RESOLVE>>"


def load_rows():
    with CSV_PATH.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    # add tracking columns if this is the first time we're running the notebook
    for row in rows:
        row.setdefault("review_status", "")       # "", "resolved", "confirmed_unknown", "skipped"
        row.setdefault("resolved_accused", "")
    return rows


def save_rows(rows):
    fieldnames = list(rows[0].keys()) if rows else []
    with CSV_PATH.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def transcript_path_for(row):
    """The .txt this row's annotation JSON was generated from."""
    return (TRANSCRIPT_ROOT / row["output_file"].replace(chr(92), os.sep)).with_suffix(".txt")


def load_transcript_lines(txt_path):
    """{line_number: 'Speaker: text'} for every '[N] ...' line in a transcript."""
    lines = {}
    for raw in txt_path.read_text(encoding="utf-8").splitlines():
        if not raw.startswith("["):
            continue
        end = raw.find("]")
        if end == -1 or not raw[1:end].isdigit():
            continue
        text = raw[end + 1:].replace(MARKER, "").strip()
        lines[int(raw[1:end])] = text
    return lines


def print_context(row, before=CONTEXT_BEFORE, after=CONTEXT_AFTER):
    """Print the surrounding dialogue, with the flagged line marked '>>>'.

    Other lines that were themselves flagged for annotation are marked with a
    trailing '*', so it's visible when several accusations sit close together.
    """
    txt_path = transcript_path_for(row)
    if not txt_path.exists():
        print(f"  (transcript not found: {txt_path})")
        return

    raw_lines = txt_path.read_text(encoding="utf-8").splitlines()
    marked = {
        int(r[1:r.find("]")])
        for r in raw_lines
        if r.startswith("[") and r.find("]") != -1
        and r[1:r.find("]")].isdigit() and MARKER in r
    }

    lines = load_transcript_lines(txt_path)
    target = int(row["line_number"])
    for n in sorted(n for n in lines if target - before <= n <= target + after):
        pointer = ">>>" if n == target else "   "
        flag = " *" if n in marked and n != target else ""
        print(f"{pointer} [{n}] {lines[n]}{flag}")


def update_annotation_json(row, new_accused):
    """Find the matching relation inside the game's output JSON and update it.
    new_accused: list[str] to resolve to, or None to just mark as manually
    confirmed while leaving accused=["UNKNOWN"] as is.
    """
    json_path = OUTPUT_ROOT / row["output_file"].replace(chr(92), os.sep)
    record = json.loads(json_path.read_text(encoding="utf-8"))

    target_line = int(row["line_number"])
    target_type = row["type"]
    target_evidence = row["evidence"]

    for item in record.get("items", []):
        if item.get("line_number") != target_line:
            continue
        for relation in item.get("relations", []):
            if relation.get("type") != target_type:
                continue
            if relation.get("accused") != ["UNKNOWN"]:
                continue
            # evidence as a tiebreaker in the rare case of duplicate type+UNKNOWN on one line
            if relation.get("evidence") != target_evidence:
                continue

            relation["manually_reviewed"] = True
            if new_accused is not None:
                relation["accused"] = new_accused
            else:
                relation["confirmed_unresolvable"] = True

            # recompute requires_review: true only if some relation on this
            # item is still an unresolved UNKNOWN
            item["requires_review"] = any(
                r.get("accused") == ["UNKNOWN"] and not r.get("confirmed_unresolvable")
                for r in item.get("relations", [])
            )

            json_path.write_text(
                json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8"
            )
            return True

    print(f"  WARNING: could not find matching relation in {json_path} "
          f"(line {target_line}, type {target_type}) -- nothing was updated.")
    return False


rows = load_rows()
pending = [r for r in rows if r["review_status"] not in ("resolved", "confirmed_unknown")]
print(f"{len(rows)} total flagged rows, {len(pending)} still pending review.")


In [ ]:
for row in pending:
    print("=" * 70)
    print(f"Game     : {row['game']}   (session: {row['session']}, {row['source']})")
    print("-" * 70)
    print_context(row)
    print("-" * 70)
    print(f"Line     : {row['line_number']}")
    print(f"Accuser  : {row['accuser']}")
    print(f"Type     : {row['type']}")
    print(f"Evidence : {row['evidence']}")
    print("-" * 70)

    answer = input(
        "Names (comma-separated) / Enter or 'u' = confirm unresolvable / "
        "'s' = skip / 'q' = quit: "
    ).strip()

    if answer.lower() == "q":
        print("Stopping. Progress so far is saved.")
        break

    if answer.lower() == "s":
        row["review_status"] = "skipped"
        save_rows(rows)
        continue

    if answer == "" or answer.lower() == "u":
        update_annotation_json(row, new_accused=None)
        row["review_status"] = "confirmed_unknown"
        save_rows(rows)
        print("-> confirmed unresolvable.")
        continue

    names = [n.strip() for n in answer.split(",") if n.strip()]
    ok = update_annotation_json(row, new_accused=names)
    if ok:
        row["review_status"] = "resolved"
        row["resolved_accused"] = ", ".join(names)
        save_rows(rows)
        print(f"-> resolved to {names}.")
    else:
        print("-> NOT saved (see warning above). Row left pending -- try again.")

print("=" * 70)
remaining = [r for r in rows if r["review_status"] not in ("resolved", "confirmed_unknown")]
print(f"Done for now. {len(remaining)} rows still pending "
      f"(skipped ones will show up again next run).")


Stopped partway, or came back later? Re-run the config cell above (it reloads the CSV with your saved progress), then run the cell below to rebuild `pending`, then re-run the loop cell.

In [ ]:
# Re-run this cell (instead of the one above) to only go through rows you
# explicitly skipped last time, without re-listing already-resolved ones.
pending = [r for r in rows if r["review_status"] not in ("resolved", "confirmed_unknown")]
print(f"{len(pending)} rows pending (including previously skipped).")
